In [1]:
!git clone https://github.com/YPolina/Medicine.git

Cloning into 'Medicine'...
remote: Enumerating objects: 1445, done.
remote: Counting objects: 100% (97/97), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 1445 (delta 41), reused 77 (delta 32), pack-reused 1348 (from 3)
Receiving objects: 100% (1445/1445), 474.77 MiB | 16.84 MiB/s, done.
Resolving deltas: 100% (115/115), done.
Updating files: 100% (103/103), done.


In [2]:
%cd ./Medicine/BELKA/training

/content/Medicine/BELKA/training


In [3]:
!pip install -r ../requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.4/311.4 MB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.3/34.3 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.4/495.4 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [16]:
import sys
import h5py
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import pandas as pd

sys.path.append(os.path.abspath(os.path.join('..')))

sys.modules.pop("functionality.models", None)
sys.modules.pop("functionality.data_preparation", None)
from functionality.data_preparation import EmbDataset, train_model
from functionality.models import ChemBertaBinaryClassifierLightning

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger
from transformers import AutoTokenizer, AutoModel

import torch
import pickle
from tqdm import tqdm
import gc
from torch.cuda.amp import autocast
import numpy as np

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
binds_0 = pd.read_parquet("../intermediates/downsampled_0_50_mln")
binds_1 = pd.read_parquet("../intermediates/1_class")
final_data = pd.concat([binds_0, binds_1], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)
del binds_1
del binds_0

In [8]:
def compute_and_save_embeddings(model_name, smiles, labels, save_path, batch_size=1000):
    """
    Compute embeddings for a list of SMILES strings in batches and save them efficiently using HDF5

    Args:
        model_name (str): Pretrained model name
        smiles (pd.Series): Data containing SMILES strings
        labels (pd.Series): Corresponding labels
        save_path (str): Path to save computed embeddings and labels
        batch_size (int): Number of SMILES strings to process per batch
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    with h5py.File(save_path, "w") as h5f:
        dset_embeddings = h5f.create_dataset("embeddings", shape=(0, 768), maxshape=(None, 768), dtype=np.float32, compression="gzip")
        dset_labels = h5f.create_dataset("labels", shape=(0,), maxshape=(None,), dtype=np.int64, compression="gzip")

        for i in tqdm(range(0, len(smiles), batch_size), desc="Computing Embeddings"):
            batch_smiles = smiles.iloc[i : i + batch_size].tolist()
            batch_labels = labels.iloc[i : i + batch_size].values.astype(np.int64)

            tokens = tokenizer(batch_smiles, padding=True, truncation=True, max_length=512, return_tensors="pt")
            tokens = {k: v.to(device) for k, v in tokens.items()}

            with torch.no_grad(), autocast():
                outputs = model(**tokens)

            batch_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()

            dset_embeddings.resize(dset_embeddings.shape[0] + batch_embeddings.shape[0], axis=0)
            dset_embeddings[-batch_embeddings.shape[0]:] = batch_embeddings

            dset_labels.resize(dset_labels.shape[0] + batch_labels.shape[0], axis=0)
            dset_labels[-batch_labels.shape[0]:] = batch_labels

            # Memory cleanup
            del batch_smiles, batch_labels, tokens, outputs, batch_embeddings
            gc.collect()
            torch.cuda.empty_cache()

    print(f"Embeddings and labels saved to {save_path}")
    return save_path

In [ ]:
protein_names = final_data.protein_name.unique()
save_dir = "/content/drive/MyDrive/embeddings/"
models = {
    "ChemBert": "seyonec/PubChem10M_SMILES_BPE_450k",
    "MolFormer": "ibm/MoLFormer-XL-both-10pct"
}
batch_size=1000

for model_name, model_ in models.items():
    for protein_name in protein_names:
        print(f"Embeddings calculations for protein: {protein_name}")

        protein_data = final_data[final_data.protein_name == protein_name]

        train_data, val_data = train_test_split(
            protein_data, test_size=0.1, random_state=42, shuffle=False
        )

        train_embeddings_path = os.path.join(save_dir, f"{protein_name}_{model_name}_train_embeddings.h5")
        val_embeddings_path = os.path.join(save_dir, f"{protein_name}_{model_name}_val_embeddings.h5")

        compute_and_save_embeddings(model_, train_data["molecule_smiles"], train_data['binds'],  train_embeddings_path, batch_size)
        compute_and_save_embeddings(model_, val_data["molecule_smiles"], val_data['binds'], val_embeddings_path, batch_size)



Embeddings calculations for protein: BRD4


Computing Embeddings:   0%|          | 0/3414 [00:00<?, ?it/s]<ipython-input-8-003d7b34f3bf>:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), autocast():
Computing Embeddings:   2%|▏         | 82/3414 [01:39<1:05:40,  1.18s/it]

In [ ]:
def train_model(final_data, models, emb_path="../intermediates/embeddings"):
    protein_names = ['sEH', 'BRD4', 'HSA']

    for model_name, model_ in models.items():
        for protein_name in protein_names:
            print(f"Training model for protein: {protein_name}")

            train_data = pd.read_parquet(os.path.join(emb_path, f"{protein_name}_{model_name}_train_data.parquet"))
            val_data = pd.read_parquet(os.path.join(emb_path, f"{protein_name}_{model_name}_val_data.parquet"))

            train_embeddings_path = os.path.join(emb_path, f"{protein_name}_{model_name}_train_embeddings.h5")
            val_embeddings_path = os.path.join(emb_path, f"{protein_name}_{model_name}_val_embeddings.h5")

            train_dataset = EmbDataset(train_data, train_embeddings_path)
            val_dataset = EmbDataset(val_data, val_embeddings_path)

            train_loader = DataLoader(train_dataset, batch_size=1000, shuffle=True, num_workers=4, pin_memory=True)
            val_loader = DataLoader(val_dataset, batch_size=1000, shuffle=False, num_workers=4, pin_memory=True)

            logger = CSVLogger("logs", name=model_name)
            early_stopping = EarlyStopping(monitor="val_loss", patience=3, mode="min")
            checkpoint_callback = ModelCheckpoint(
                dirpath="../checkpoints",
                filename=f"{model_name}_{protein_name}-{{epoch}}-{{val_loss:.4f}}",
                monitor="val_loss",
                save_top_k=1,
                mode="min",
                save_last=True,
                verbose=True,
            )

            trainer = pl.Trainer(
                max_epochs=20,
                accelerator="auto",
                devices=1,
                log_every_n_steps=2,
                callbacks=[early_stopping, checkpoint_callback],
                logger=logger,
            )


            chemberta_model = ChemBertaBinaryClassifierLightning()
            trainer.fit(chemberta_model, train_loader, val_loader)

            os.makedirs("../intermediates/models", exist_ok=True)
            trainer.save_checkpoint(f"../intermediates/models/{model_name}_{protein_name}.ckpt")

            print(f"Completed training for protein: {protein_name}")

            del train_data, val_data, train_dataset, val_dataset, train_loader, val_loader, chemberta_model
            gc.collect()

In [ ]:
train_model(final_data)

Training model for protein: BRD4


Computing Embeddings:   0%|          | 0/342 [00:00<?, ?it/s]

: 